# Faster R-CNN multiscale training on a GPU

Run the existing `vehicle_detection.training.faster_rcnn_train` implementation on **Google Colab or Kaggle**. This notebook downloads the real TRANSSET dataset directly from Figshare; S3 credentials are unnecessary.

Before running:
1. Enable an NVIDIA GPU in the notebook runtime settings. On Kaggle, enable Internet too.
2. Make sure the selected Git branch contains `src/vehicle_detection/training/faster_rcnn_train.py`, its configuration, and `splits/v1.csv`. Commit and push local changes first, or upload a ZIP of the current project source.
3. Run the cells in order, including validation threshold selection and final test comparison after training. The training target is 50 epochs; set EPOCHS to 1 if you want a short timing run first.

The trainer uses one GPU, even if the service provides several. GPU availability and session duration depend on the service. Save outputs before a session ends.

In [ ]:
import os
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path

import torch
import torchvision
from torchvision.ops import nms

IS_COLAB = "google.colab" in sys.modules
BASE = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")
if not BASE.exists():
    BASE = Path.cwd()

# Use the branch containing your Faster R-CNN implementation.
REPO_URL = "https://github.com/FapaKslapa/vehicle-detection.git"
BRANCH = "development"
# For a private repository or unpushed work, upload a SOURCE-ONLY project ZIP
# and set its absolute path here. Exclude .venv, data, .git, outputs and secrets.
SOURCE_ZIP = None  # Example: "/content/vehicle-detection.zip"
WORK = BASE / "faster-rcnn-project"

EPOCHS = 50  # Total epoch target, including completed epochs when resuming.
BATCH_SIZE = 2  # Reduce to 1 if GPU memory is insufficient.
WORKERS = 0
SEED = 0
MIN_SIZES = [480, 576, 672, 768, 800]
MAX_SIZE = 1333
RESUME_CHECKPOINT = None  # Absolute path to checkpoint.pt, if resuming.
USE_GOOGLE_DRIVE = False  # Colab only: persist outputs directly to Drive.


def run(args, **kwargs):
    return subprocess.run([str(arg) for arg in args], check=True, **kwargs)


assert torch.cuda.is_available(), "Enable an NVIDIA GPU and restart the runtime before continuing."
print("Python:", sys.version.split()[0])
assert (3, 11) <= sys.version_info[:2] < (3, 14), "This project supports Python 3.11-3.13."
print("PyTorch:", torch.__version__, "torchvision:", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
# Verify that the preinstalled PyTorch and torchvision compiled operators match.

nms(torch.tensor([[0.0, 0.0, 10.0, 10.0]], device="cuda"), torch.tensor([0.9], device="cuda"), 0.5)

## Load the project

The default clone works for a publicly readable repository. For a private repository, upload a source ZIP and set `SOURCE_ZIP` above. Do not place GitHub tokens in notebook cells or repository URLs. Reruns reuse the existing checkout without resetting it.

In [ ]:
import zipfile


def safe_extract(archive, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(archive) as zipped:
        for entry in zipped.infolist():
            target = (destination / entry.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError(f"Unsafe ZIP entry: {entry.filename}")
        zipped.extractall(destination)


if not WORK.exists():
    if SOURCE_ZIP:
        unpacked = BASE / "uploaded-frcnn-source"
        safe_extract(SOURCE_ZIP, unpacked)
        candidates = [
            p.parent.parent for p in unpacked.rglob("src/vehicle_detection") if p.is_dir()
        ]
        if len(candidates) != 1:
            raise RuntimeError(
                "The source ZIP must contain exactly one project with src/vehicle_detection."
            )
        import shutil

        shutil.copytree(candidates[0], WORK)
    else:
        run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, WORK],
            env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
        )

required = [
    "src/vehicle_detection/training/faster_rcnn_train.py",
    "configs/train/faster_rcnn.yaml",
    "splits/v1.csv",
]
missing = [name for name in required if not (WORK / name).is_file()]
assert not missing, (
    f"Missing project files: {missing}. Push your changes or use a current source ZIP."
)
os.chdir(WORK)
os.environ["PYTHONPATH"] = str(WORK / "src")
if str(WORK / "src") not in sys.path:
    sys.path.insert(0, str(WORK / "src"))
print("Project:", WORK)

## Install training dependencies

Use the runtime's GPU-enabled PyTorch and torchvision. Install only the additional libraries needed by this trainer and its shared imports; the full project also has unrelated tools such as DVC and W&B. No W&B account is required here.

In [ ]:
run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "hydra-core",
        "pandas",
        "matplotlib",
        "pillow",
        "pycocotools",
        "sahi",
        "opencv-python-headless",
        "pydantic>=2",
        "tqdm",
    ]
)
# Check imports in the same type of fresh process used for training.
run(
    [
        sys.executable,
        "-c",
        "import torch; import vehicle_detection.training.faster_rcnn_train; "
        "assert torch.cuda.is_available(); print('Trainer import and CUDA check passed')",
    ]
)

## Download and prepare real TRANSSET data

Source: https://doi.org/10.6084/m9.figshare.26082217

This uses the verified `TRANSSET_v2.zip` Figshare file (741,382,444 bytes). The archive is checksum-checked before extraction. The project normalizer adds image filename extensions, removes invalid boxes, and clips boundary overshoots. The existing team split is retained without reshuffling.

In [ ]:
import hashlib
import urllib.request

from tqdm.auto import tqdm

from vehicle_detection.data.normalise import normalise_coco

ARCHIVE = WORK / "data/downloads/TRANSSET_v2.zip"
EXPECTED_MD5 = "3afac47a3a4f9f930f43a109a37fc78f"
DOWNLOAD_URL = "https://ndownloader.figshare.com/files/62024113"
IMAGES = WORK / "data/raw/transset/TRANSSET_v2/images"
RAW_COCO = IMAGES.parent / "label_coco/coco.json"
COCO = WORK / "data/processed/coco.json"
MANIFEST = WORK / "splits/v1.csv"


def md5_file(path):
    checksum = hashlib.md5()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            checksum.update(block)
    return checksum.hexdigest()


ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
if not ARCHIVE.exists():
    partial = ARCHIVE.with_suffix(".zip.part")
    with (
        urllib.request.urlopen(DOWNLOAD_URL, timeout=120) as response,
        partial.open("wb") as output,
        tqdm(
            total=int(response.headers.get("Content-Length", 741382444)), unit="B", unit_scale=True
        ) as progress,
    ):
        while block := response.read(1024 * 1024):
            output.write(block)
            progress.update(len(block))
    assert md5_file(partial) == EXPECTED_MD5, "Checksum mismatch. Remove the .part file and retry."
    partial.replace(ARCHIVE)
assert md5_file(ARCHIVE) == EXPECTED_MD5, (
    "Cached archive checksum mismatch; remove the archive and retry."
)
# Extraction is repeatable and repairs a previously interrupted extraction.
safe_extract(ARCHIVE, WORK / "data/raw/transset")

report = normalise_coco(RAW_COCO, IMAGES, COCO)
print("Normalized annotations:", report)

In [ ]:
from PIL import Image

from vehicle_detection.data.coco_split import load_coco, split_coco_by_manifest
from vehicle_detection.data.manifest import load_manifest
from vehicle_detection.training.faster_rcnn_train import CocoDataset

coco = load_coco(COCO)
subsets = split_coco_by_manifest(coco, load_manifest(MANIFEST))
for record in tqdm(coco["images"], desc="Checking images"):
    with Image.open(IMAGES / record["file_name"]) as picture:
        assert picture.size == (record["width"], record["height"]), record["file_name"]
        picture.verify()
assert subsets["train"]["images"] and subsets["val"]["images"]
print("Split counts:", {name: len(part["images"]) for name, part in subsets.items()})
print("Actual COCO categories:", [(c["id"], c["name"]) for c in coco["categories"]])
sample, target = CocoDataset(subsets["train"], IMAGES)[0]
print("Sample:", tuple(sample.shape), "boxes:", tuple(target["boxes"].shape))
# The trainer derives labels from COCO; the legacy data.classes list is unused.

## Check a complete real GPU training step

This runs one batch of real images through Faster R-CNN at the configured multiscale sizes: zero gradients, forward pass, finite loss check, backward pass, and an SGD optimizer update. It checks finite, nonzero prediction-head gradients and confirms that prediction-head weights change. It uses random weights to avoid a second pretrained download. The temporary model is discarded; the separate 50-epoch training run starts from COCO-pretrained weights. This single-batch check is not a completed epoch or a full training run.

In [ ]:
import gc

from vehicle_detection.training.faster_rcnn import setup_faster_rcnn

data = CocoDataset(subsets["train"], IMAGES)
smoke_model = setup_faster_rcnn(
    len(coco["categories"]),
    weights=None,
    weights_backbone=None,
    device=torch.device("cuda"),
    min_size=MIN_SIZES,
    max_size=MAX_SIZE,
).train()
smoke_samples = [data[index] for index in range(BATCH_SIZE)]
smoke_images = [image.cuda() for image, _ in smoke_samples]
smoke_targets = [
    {key: value.cuda() for key, value in target.items()} for _, target in smoke_samples
]
smoke_optimizer = torch.optim.SGD(smoke_model.parameters(), lr=0.001, momentum=0.9)
smoke_head = smoke_model.roi_heads.box_predictor.cls_score.weight
smoke_before = smoke_head.detach().clone()
smoke_optimizer.zero_grad(set_to_none=True)
smoke_losses = smoke_model(smoke_images, smoke_targets)
smoke_loss = sum(smoke_losses.values())
assert torch.isfinite(smoke_loss), smoke_losses
smoke_loss.backward()
assert smoke_head.grad is not None, "Prediction head received no gradients"
assert torch.isfinite(smoke_head.grad).all(), "Non-finite prediction-head gradients"
assert torch.count_nonzero(smoke_head.grad) > 0, "Prediction-head gradients are zero"
smoke_optimizer.step()
assert not torch.equal(smoke_head.detach(), smoke_before), "Weights did not update"
print(
    "GPU training step passed (forward, backward, weight update):",
    {key: float(value.detach()) for key, value in smoke_losses.items()},
)
del smoke_optimizer, smoke_head, smoke_before
del smoke_model, smoke_images, smoke_targets, smoke_losses, smoke_loss, smoke_samples

gc.collect()
torch.cuda.empty_cache()

## Train and save checkpoints

The first run defaults to **50 epochs**. The existing trainer prints metrics only after training and validation complete for each epoch. It saves `best.pt`, `last.pt`, `checkpoint.pt`, `metrics.jsonl`, and `run.json`.

On Colab, set `USE_GOOGLE_DRIVE=True` before this cell to save directly to Drive. Otherwise outputs are on temporary runtime storage. On Kaggle, preserve/download the output files before the session ends.

To resume: retain the full previous run folder, set `RESUME_CHECKPOINT` to its checkpoint, and increase `EPOCHS` to the total desired count. Use the same model, seed, batch size, learning rate, and data settings. This trainer requires a **new output folder** on resume and selects the best model within that new segment. Compare validation AP across segments to retain the overall best weights. Interruptions before the first epoch finishes produce no checkpoint.

In [ ]:
import time

if USE_GOOGLE_DRIVE:
    if not IS_COLAB:
        raise RuntimeError("Google Drive mounting is supported here only in Colab.")
    from google.colab import drive

    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/vehicle-detection-runs")
else:
    OUTPUT_ROOT = BASE / "frcnn-runs"

RUN_ID = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
OUT = OUTPUT_ROOT / f"frcnn-multiscale-{RUN_ID}"
command = [
    sys.executable,
    "-u",
    "-m",
    "vehicle_detection.training.faster_rcnn_train",
    f"data.coco_path={COCO}",
    f"data.images_dir={IMAGES}",
    f"data.manifest_path={MANIFEST}",
    f"training.output_dir={OUT}",
    f"training.epochs={EPOCHS}",
    f"training.batch_size={BATCH_SIZE}",
    f"training.workers={WORKERS}",
    f"training.seed={SEED}",
    "training.device=cuda",
    f"model.min_size={MIN_SIZES}",
    f"model.max_size={MAX_SIZE}",
]
if RESUME_CHECKPOINT:
    checkpoint_path = Path(RESUME_CHECKPOINT).expanduser().resolve()
    assert checkpoint_path.is_file(), checkpoint_path
    command.append(f"training.resume={checkpoint_path}")
print("Output directory:", OUT)
print("Epoch target:", EPOCHS)

started = time.monotonic()
run(command, cwd=WORK)
print("Elapsed minutes:", round((time.monotonic() - started) / 60, 1))

## Inspect and download results

Training and model selection use train and validation only. Keep the test split reserved for final evaluation. Use completed epoch durations to estimate the remaining time; runtime availability can still interrupt the 50-epoch run.

In [ ]:
import pandas as pd

history_path = OUT / "metrics.jsonl"
assert history_path.is_file(), "No epoch metrics saved yet. Check the training cell for errors."
history = pd.read_json(history_path, lines=True)
display(history)
axes = history.plot(
    x="epoch",
    y=["train_loss", "val_ap"],
    subplots=True,
    marker="o",
    figsize=(9, 6),
    title=["Training loss", "Validation COCO AP"],
)
axes[-1].figure.tight_layout()
print("Files:")
for file in sorted(OUT.iterdir()):
    print(file.name, f"{file.stat().st_size / 2**20:.1f} MiB")

## Evaluate best weights with the shared compare pipeline

Training metrics are not final test results. After training completes, reload this run's
`best.pt` and use `vehicle_detection.experiments.compare`, the same pipeline used for
E1 and E3b. First tune each inference variant's counting threshold on **validation**.
Then freeze those thresholds and evaluate **test once**, without choosing weights,
thresholds, or inference settings from test results. The variants are full, sliced,
sliced with standard prediction, and adaptive, using the shared compare defaults.

The compare adapter reconstructs Faster R-CNN with its standard evaluation resize
(800 short edge, 1333 long-edge cap). This matches the default training configuration's
evaluation transform. Custom training sizes do not change this adapter's resize.
The reports include COCO AP/AP50 and size breakdowns, precision/recall/F1, counting
MAE/RMSE, inference time, peak memory, per-class metrics, predictions and provenance.
Reports are saved under this run's `compare_val/` and `compare_test/` folders and
included in the results archive. A full completed training run is required first.


In [ ]:
import json
import math

from hydra import compose, initialize_config_dir

from vehicle_detection.experiments.compare import run as run_compare

BEST_WEIGHTS = OUT / "best.pt"
assert BEST_WEIGHTS.is_file(), "Missing best.pt: complete training first."
assert int(history["epoch"].max()) == EPOCHS, "Complete the planned epochs before final evaluation."
training_provenance = json.loads((OUT / "run.json").read_text())
assert (
    hashlib.sha256(MANIFEST.read_bytes()).hexdigest() == training_provenance["manifest_sha256"]
), "Manifest changed since training; use the original split."
assert subsets.get("test", {}).get("images"), "The manifest must contain a nonempty test split."
VAL_REPORT = OUT / "compare_val"
TEST_REPORT = OUT / "compare_test"
assert not VAL_REPORT.exists() and not TEST_REPORT.exists(), (
    "Evaluation reports already exist. Preserve them; do not repeat test-driven tuning."
)
with initialize_config_dir(version_base=None, config_dir=str(WORK / "configs")):
    compare_cfg = compose(config_name="compare")
compare_cfg.kind = "torchvision"
compare_cfg.torchvision_name = "fasterrcnn_resnet50_fpn"
compare_cfg.weights = str(BEST_WEIGHTS)
compare_cfg.coco_path = str(COCO)
compare_cfg.images_dir = str(IMAGES)
compare_cfg.manifest_path = str(MANIFEST)
compare_cfg.device = "cuda"
compare_cfg.split = "val"
compare_cfg.out_dir = str(VAL_REPORT)
compare_cfg.limit = None
compare_cfg.threshold = None
print("Selecting thresholds on validation using:", BEST_WEIGHTS)
validation_comparison = run_compare(compare_cfg)
threshold_by_variant = dict(
    zip(validation_comparison["variant"], validation_comparison["threshold"], strict=True)
)
assert all(math.isfinite(value) for value in threshold_by_variant.values())
display(validation_comparison)
print("Frozen validation thresholds:", threshold_by_variant)

In [ ]:
from omegaconf import OmegaConf

# Copy the exact validation inference configuration; only split, output directory,
# and the validation-selected fixed thresholds change. Never tune on test.
assert not TEST_REPORT.exists(), "Test reports already exist; inspect the saved results."
test_cfg = OmegaConf.create(OmegaConf.to_container(compare_cfg, resolve=True))
test_cfg.split = "test"
test_cfg.out_dir = str(TEST_REPORT)
for variant in test_cfg.variants:
    variant.threshold = float(threshold_by_variant[variant.name])
test_comparison = run_compare(test_cfg)
assert (test_comparison["threshold_source"] == "fixed").all()
display(test_comparison)
display(pd.read_csv(TEST_REPORT / "per_class.csv"))
print("Final test report:", TEST_REPORT / "comparison.md")
print("Predictions and provenance saved alongside the report.")

In [ ]:
import shutil

archive_path = Path(
    shutil.make_archive(str(BASE / OUT.name), "zip", root_dir=OUT.parent, base_dir=OUT.name)
)
print("Results archive:", archive_path)
print("Keep run.json alongside weights so the real categories and configuration remain available.")
if IS_COLAB:
    from google.colab import files

    files.download(str(archive_path))
else:
    from IPython.display import FileLink, display

    display(FileLink(str(archive_path)))
    print("On Kaggle, you can also download this archive from the notebook Output files.")

## Troubleshooting

- **No CUDA:** enable a GPU runtime, restart, and rerun from the first cell.
- **Missing trainer module:** push the current implementation to the chosen branch or upload a current source ZIP. The module is `faster_rcnn_train`, not `frcnn_train`.
- **GPU out of memory:** restart the runtime, set `BATCH_SIZE=1`, and rerun. If necessary reduce `MIN_SIZES` and `MAX_SIZE`, recording the change as a different experiment. A failed training launch may leave an output directory; rerunning the training cell creates a new one.
- **Long silence:** this trainer reports once per completed epoch, after validation. GPU use can be inspected with `nvidia-smi` in another cell or terminal.
- **Session ends:** resume from the most recent saved checkpoint. Progress within an unfinished epoch is lost. Download and preserve artifacts regularly.
- **Pretrained download:** the first real training run downloads about 160 MB of COCO-pretrained detector weights.
- **Resume into the same folder fails:** this project's trainer requires a new output folder. The timestamped folder above handles this automatically.